In [1]:
import weakref
import torch
from dataclasses import dataclass
from collections import OrderedDict
from typing import Tuple

In [2]:
_finalizers = {}
REG_PARAM = OrderedDict()

In [3]:
@dataclass
class TensorInfo:
    tensor_id: int  # id(tensor), cannot store tensor directly, because it will prevent tensor from being released
    canonical_name: str
    full_shape: Tuple[int, ...]
    shard_slice: Tuple[slice, ...]
    dtype: torch.dtype
    # device: torch.device
    local_name: str = ""  # don't know what to do with it...

In [4]:
def _cleanup_callback(tensor_id):
    """
    Callback to remove the tensor and finalizer from the registry to release memory.
    """
    canonical_name = REG_PARAM[tensor_id].canonical_name
    print(f"Cleaning up tensor {canonical_name}")
    del REG_PARAM[canonical_name]
    del REG_PARAM[tensor_id]
    del _finalizers[tensor_id]

In [5]:
def register_tensor( 
        tensor: torch.Tensor,
        tensor_info: TensorInfo, 
        destination: OrderedDict
    ) -> None:
    """
    Register a tensor with a name.
    Need this functionality because cannot add a field to the tensor (will lose in t.to())
    """
    tid = tensor_info.tensor_id
    destination[tensor_info.canonical_name] = tensor_info
    destination[tid] = tensor_info
    _finalizers[tid] = weakref.finalize(tensor, _cleanup_callback, tid)

In [6]:
import torch.nn as nn 
class MyModel(nn.Module): 
    def __init__(self): 
        super(MyModel, self).__init__() 
        self.weight = nn.Parameter(torch.randn(3, 3)) 
        self.bias = nn.Parameter(torch.randn(3)) 
        
    def forward(self, x): 
        return x @ self.weight + self.bias 

In [7]:
model = MyModel()

In [8]:
for name, param in model.named_parameters():
    tensor_info = TensorInfo(
        tensor_id=id(param),
        canonical_name=name,
        full_shape=param.shape,
        shard_slice=(slice(None),),
        dtype=param.dtype
    )
    register_tensor(param, tensor_info, REG_PARAM)

In [9]:
_finalizers

{140431836085392: <finalize object at 0x7fb80edd1c60; for 'Parameter' at 0x7fb8d5b47c90>,
 140428500056992: <finalize object at 0x7fb80edd1b60; for 'Parameter' at 0x7fb80edcbba0>}

In [10]:
REG_PARAM

OrderedDict([('weight',
              TensorInfo(tensor_id=140431836085392, canonical_name='weight', full_shape=torch.Size([3, 3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name='')),
             (140431836085392,
              TensorInfo(tensor_id=140431836085392, canonical_name='weight', full_shape=torch.Size([3, 3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name='')),
             ('bias',
              TensorInfo(tensor_id=140428500056992, canonical_name='bias', full_shape=torch.Size([3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name='')),
             (140428500056992,
              TensorInfo(tensor_id=140428500056992, canonical_name='bias', full_shape=torch.Size([3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name=''))])

In [11]:
del model.weight

Cleaning up tensor weight


In [12]:
del model.bias

In [15]:
REG_PARAM

OrderedDict([('bias',
              TensorInfo(tensor_id=130330982503008, canonical_name='bias', full_shape=torch.Size([3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name='')),
             (130330982503008,
              TensorInfo(tensor_id=130330982503008, canonical_name='bias', full_shape=torch.Size([3]), shard_slice=(slice(None, None, None),), dtype=torch.float32, local_name=''))])

In [16]:
_finalizers

{130330982503008: <finalize object at 0x76890beea620; for 'Parameter' at 0x76890bea5260>}

In [ ]:
import sys
import ctypes
gc.get_referrers(ctypes.cast(130330982503008, ctypes.py_object).value)